# MML Chapter 8: When Models Meet Data

> _Data is observed; models are assumed; parameters are learned._

This notebook accompanies Chapter 8 of *Mathematics for Machine Learning* (Deisenroth, Faisal, Ong). Each section runs a concrete experiment to build intuition for the key ideas:

1. **Loss functions** — MSE, MAE, Huber: trade-offs in practice
2. **Bias-variance tradeoff** — polynomial regression on noisy sine data
3. **MLE for a Gaussian** — derive and verify on generated data
4. **MLE = NLL minimization** — numerical verification of the equivalence
5. **Bayesian vs. frequentist** — confidence intervals vs. credible intervals for a coin
6. **BIC model selection** — choosing polynomial degree via BIC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from scipy.special import betaln

np.random.seed(42)

# Consistent color palette
C = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Setup complete.')

---
## 1. Loss Functions: MSE, MAE, and Huber

The loss function $\ell(\hat{y}, y)$ measures how wrong a single prediction is. Three classical choices:

| Name | Formula | Key property |
|------|---------|-------------|
| MSE (squared loss) | $(\hat{y} - y)^2$ | Differentiable everywhere; heavily penalizes large errors |
| MAE (absolute loss) | $|\hat{y} - y|$ | Robust to outliers; non-differentiable at 0 |
| Huber loss | $\frac{1}{2}r^2$ if $|r|\le\delta$, else $\delta|r|-\frac{\delta^2}{2}$ | Quadratic near 0, linear in tails — best of both |

where $r = \hat{y} - y$ is the residual.

**When to use which:**
- **MSE**: default for regression; assumes Gaussian noise; the squared penalty means a single outlier at $r=10$ contributes as much as 100 points at $r=1$.
- **MAE**: prefer when data contains outliers (heavy-tailed noise); corresponds to Laplace likelihood. Non-differentiability requires subgradient or smoothed variants.
- **Huber**: practical compromise. Behaves like MSE for small residuals (good gradient signal) and like MAE for large ones (outlier robustness). $\delta$ is a tunable threshold.

In [ ]:
def mse_loss(r):
    return r**2

def mae_loss(r):
    return np.abs(r)

def huber_loss(r, delta=1.0):
    return np.where(
        np.abs(r) <= delta,
        0.5 * r**2,
        delta * np.abs(r) - 0.5 * delta**2
    )

r = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: loss values
ax = axes[0]
ax.plot(r, mse_loss(r), label='MSE', lw=2)
ax.plot(r, mae_loss(r), label='MAE', lw=2, linestyle='--')
ax.plot(r, huber_loss(r, delta=1.0), label='Huber (δ=1)', lw=2, linestyle=':')
ax.set_xlabel('Residual  r = ŷ − y')
ax.set_ylabel('Loss ℓ(r)')
ax.set_title('Loss Functions')
ax.set_ylim(-0.2, 8)
ax.legend()

# Right: gradients (derivatives w.r.t. r)
ax2 = axes[1]
ax2.plot(r, 2*r, label='∂MSE/∂r = 2r', lw=2)
ax2.plot(r, np.sign(r), label='∂MAE/∂r = sign(r)', lw=2, linestyle='--')
grad_huber = np.where(np.abs(r) <= 1.0, r, np.sign(r))
ax2.plot(r, grad_huber, label='∂Huber/∂r', lw=2, linestyle=':')
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel('Residual  r')
ax2.set_ylabel('Gradient')
ax2.set_title('Gradients of Loss Functions')
ax2.set_ylim(-3, 3)
ax2.legend()

plt.tight_layout()
plt.suptitle('Fig 1: Loss Functions and Their Gradients', y=1.02, fontsize=13)
plt.show()

# Illustrate outlier sensitivity
print("--- Outlier sensitivity demo ---")
outlier_r = 5.0
print(f"Residual = {outlier_r}")
print(f"  MSE   : {mse_loss(outlier_r):.1f}  (25x larger than at r=1)")
print(f"  MAE   : {mae_loss(outlier_r):.1f}  (5x larger than at r=1)")
print(f"  Huber : {huber_loss(outlier_r, 1.0):.1f}  (linear, capped)")

---
## 2. Bias-Variance Tradeoff

For squared loss, the expected test error decomposes as:

$$\mathbb{E}_\mathcal{D}[(y - \hat{f}(x))^2] = \underbrace{\text{Bias}^2}_\text{underfitting} + \underbrace{\text{Var}}_\text{overfitting} + \underbrace{\sigma^2}_\text{irreducible}$$

**Experiment**: Fit polynomial regressors of degree $d = 1, 5, 10$ to noisy sine data. Track train and test MSE as $d$ grows.

In [ ]:
# Generate noisy sine data
def true_fn(x):
    return np.sin(2 * np.pi * x)

N_train, N_test = 20, 200
noise_std = 0.3

x_train = np.sort(np.random.uniform(0, 1, N_train))
y_train = true_fn(x_train) + np.random.normal(0, noise_std, N_train)

x_test = np.linspace(0, 1, N_test)
y_test = true_fn(x_test) + np.random.normal(0, noise_std, N_test)

degrees = list(range(1, 12))
train_errors, test_errors = [], []

for d in degrees:
    coeffs = np.polyfit(x_train, y_train, d)
    poly = np.poly1d(coeffs)
    train_errors.append(np.mean((y_train - poly(x_train))**2))
    test_errors.append(np.mean((y_test - poly(x_test))**2))

# --- Plots ---
highlight_degrees = [1, 5, 10]
x_fine = np.linspace(-0.05, 1.05, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: fitted curves
ax = axes[0]
ax.scatter(x_train, y_train, s=30, color='k', zorder=5, label='Train data', alpha=0.7)
ax.plot(x_fine, true_fn(x_fine), 'k--', lw=1.5, label='True sin(2πx)', alpha=0.5)
for d in highlight_degrees:
    coeffs = np.polyfit(x_train, y_train, d)
    ax.plot(x_fine, np.poly1d(coeffs)(x_fine), lw=2, label=f'degree {d}')
ax.set_ylim(-3, 3)
ax.set_xlim(-0.05, 1.05)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Polynomial Fits to Noisy Sine Data')
ax.legend(fontsize=9)

# Right: train/test error vs degree
ax2 = axes[1]
ax2.plot(degrees, train_errors, 'o-', label='Train MSE', lw=2)
ax2.plot(degrees, test_errors, 's--', label='Test MSE', lw=2)
ax2.axhline(noise_std**2, color='grey', linestyle=':', lw=1.5, label=f'Noise floor σ²={noise_std**2:.2f}')
for d in highlight_degrees:
    ax2.axvline(d, color='grey', alpha=0.4, lw=1)
    ax2.text(d + 0.1, max(test_errors)*0.9, f'd={d}', fontsize=9, color='grey')
ax2.set_xlabel('Polynomial degree d')
ax2.set_ylabel('MSE')
ax2.set_title('Train vs Test Error (Bias-Variance)')
ax2.legend()
ax2.set_yscale('log')

plt.tight_layout()
plt.suptitle('Fig 2: Bias-Variance Tradeoff via Polynomial Regression', y=1.02, fontsize=13)
plt.show()

print(f"Degree  1 — Train MSE: {train_errors[0]:.4f}  Test MSE: {test_errors[0]:.4f}  (HIGH BIAS / underfitting)")
print(f"Degree  5 — Train MSE: {train_errors[4]:.4f}  Test MSE: {test_errors[4]:.4f}  (good balance)")
print(f"Degree 10 — Train MSE: {train_errors[9]:.4f}  Test MSE: {test_errors[9]:.4f}  (HIGH VARIANCE / overfitting)")

---
## 3. MLE for a Gaussian

**Setup**: $x_i \overset{\text{i.i.d.}}{\sim} \mathcal{N}(\mu, \sigma^2)$ with $\mu = 3$, $\sigma^2 = 2$.

**Log-likelihood**:
$$\log L(\mu, \sigma^2 \mid \mathcal{D}) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^N (x_i - \mu)^2$$

Setting derivatives to zero yields closed-form MLEs:
$$\hat\mu_{\text{MLE}} = \frac{1}{N}\sum x_i, \qquad \hat\sigma^2_{\text{MLE}} = \frac{1}{N}\sum(x_i - \hat\mu)^2$$

Note: $\hat\sigma^2_{\text{MLE}}$ divides by $N$ (biased); the unbiased estimator divides by $N-1$.

In [ ]:
true_mu, true_sigma2 = 3.0, 2.0
N = 200

data = np.random.normal(true_mu, np.sqrt(true_sigma2), N)

# Analytic MLE
mu_mle = np.mean(data)
sigma2_mle = np.mean((data - mu_mle)**2)          # biased (divides by N)
sigma2_unbiased = np.var(data, ddof=1)             # unbiased (divides by N-1)

# Verify against numpy
mu_np = np.mean(data)
sigma2_np = np.var(data, ddof=0)  # ddof=0 → MLE

print(f"True parameters :  μ = {true_mu},  σ² = {true_sigma2}")
print(f"MLE estimates   :  μ̂ = {mu_mle:.4f},  σ̂² = {sigma2_mle:.4f}")
print(f"NumPy (ddof=0)  :  μ = {mu_np:.4f},  σ² = {sigma2_np:.4f}  ← matches MLE exactly")
print(f"Unbiased (N-1)  :  σ² = {sigma2_unbiased:.4f}  (slightly larger — Bessel's correction)")
print(f"MLE bias in σ²  :  {sigma2_mle - sigma2_unbiased:.4f}  (negative → MLE underestimates)")

# Visualize: likelihood surface over (mu, sigma2) grid
def log_likelihood_gauss(mu_val, sigma2_val, x):
    N = len(x)
    if sigma2_val <= 0:
        return -np.inf
    return (-N/2) * np.log(2*np.pi*sigma2_val) - (1/(2*sigma2_val)) * np.sum((x - mu_val)**2)

mu_grid = np.linspace(2.0, 4.0, 80)
s2_grid = np.linspace(0.5, 4.0, 80)
MU, S2 = np.meshgrid(mu_grid, s2_grid)
LL = np.array([[log_likelihood_gauss(m, s, data) for m in mu_grid] for s in s2_grid])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Log-likelihood surface
ax = axes[0]
cf = ax.contourf(MU, S2, LL, levels=30, cmap='viridis')
plt.colorbar(cf, ax=ax, label='log L')
ax.plot(mu_mle, sigma2_mle, 'r*', markersize=14, label='MLE estimate')
ax.plot(true_mu, true_sigma2, 'w+', markersize=14, markeredgewidth=2, label='True params')
ax.set_xlabel('μ')
ax.set_ylabel('σ²')
ax.set_title('Log-Likelihood Surface  log L(μ, σ²)')
ax.legend(fontsize=9)

# Data histogram vs fitted Gaussian
ax2 = axes[1]
xs = np.linspace(data.min() - 1, data.max() + 1, 200)
ax2.hist(data, bins=30, density=True, alpha=0.5, label='Data (N=200)')
ax2.plot(xs, stats.norm.pdf(xs, true_mu, np.sqrt(true_sigma2)),
         'k--', lw=2, label=f'True  N(μ={true_mu}, σ²={true_sigma2})')
ax2.plot(xs, stats.norm.pdf(xs, mu_mle, np.sqrt(sigma2_mle)),
         'r-', lw=2, label=f'MLE   N({mu_mle:.2f}, {sigma2_mle:.2f})')
ax2.set_xlabel('x')
ax2.set_ylabel('Density')
ax2.set_title('Data vs MLE-Fitted Gaussian')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Fig 3: MLE for a Gaussian', y=1.02, fontsize=13)
plt.show()

---
## 4. MLE = NLL Minimization

The key bridge in §8.3: **maximizing the log-likelihood is identical to minimizing the negative log-likelihood (NLL)**.

$$\hat\theta_{\text{MLE}} = \arg\max_\theta \sum_i \log p(x_i | \theta) = \arg\min_\theta \left[-\frac{1}{N}\sum_i \log p(x_i | \theta)\right]$$

Below we verify this numerically by:
1. Scanning $\mu$ and plotting both $\log L(\mu)$ and $\text{NLL}(\mu)$.
2. Running a numerical optimizer on the NLL and confirming it recovers the same $\hat\mu$ as the analytic formula.

In [ ]:
# Fix sigma2 at its MLE value; sweep mu
sigma2_fixed = sigma2_mle
mu_scan = np.linspace(1.5, 4.5, 300)

def neg_log_lik(mu_val):
    """NLL for Gaussian with fixed sigma2, as a function of mu."""
    return 0.5 * np.log(2 * np.pi * sigma2_fixed) + (1 / (2 * sigma2_fixed)) * np.mean((data - mu_val)**2)

log_lik_scan = np.array([-neg_log_lik(m) for m in mu_scan])  # log L (positive = higher is better)
nll_scan     = np.array([ neg_log_lik(m) for m in mu_scan])  # NLL   (lower is better)

# Numerical optimizer on NLL
result = optimize.minimize_scalar(neg_log_lik, bounds=(1.0, 5.0), method='bounded')
mu_numerical = result.x

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(mu_scan, log_lik_scan, lw=2, label='Log-likelihood log L(μ)')
ax.axvline(mu_mle, color='red', linestyle='--', lw=1.5, label=f'MLE  μ̂={mu_mle:.3f}')
ax.set_xlabel('μ')
ax.set_ylabel('log L(μ)  (higher = better)')
ax.set_title('Maximizing Log-Likelihood')
ax.legend()

ax2 = axes[1]
ax2.plot(mu_scan, nll_scan, lw=2, color=C[1], label='NLL −log L(μ)/N')
ax2.axvline(mu_numerical, color='red', linestyle='--', lw=1.5,
            label=f'Numerical min  μ={mu_numerical:.3f}')
ax2.set_xlabel('μ')
ax2.set_ylabel('NLL  (lower = better)')
ax2.set_title('Minimizing Negative Log-Likelihood')
ax2.legend()

plt.tight_layout()
plt.suptitle('Fig 4: MLE = NLL Minimization (two sides of the same coin)', y=1.02, fontsize=13)
plt.show()

print(f"Analytic MLE         :  μ̂ = {mu_mle:.6f}")
print(f"Numerical NLL argmin :  μ̂ = {mu_numerical:.6f}")
print(f"Difference           :  {abs(mu_mle - mu_numerical):.2e}  (numerical precision)")

---
## 5. Bayesian vs. Frequentist: Coin Flip

**Frequentist**: $p$ is a fixed unknown. After $N$ flips with $k$ heads, a 95% confidence interval is:
$$\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/N}, \quad \hat p = k/N$$

**Bayesian**: $p$ is a random variable. With a uniform prior $p \sim \text{Beta}(1, 1)$, the posterior after observing $k$ heads in $N$ flips is:
$$p \mid k, N \sim \text{Beta}(k+1, N-k+1)$$

A 95% **credible interval** is the central 95% of this posterior.

**Key conceptual difference**:
- The frequentist CI is a statement about the procedure: if we repeated the experiment many times, 95% of such intervals would contain the true $p$. $p$ itself has no distribution.
- The Bayesian CI (credible interval) is a direct probability statement about $p$: given the data, $P(p \in [L, U]) = 0.95$.

In [ ]:
true_p = 0.6
N_max = 200
flips = np.random.binomial(1, true_p, N_max)  # 1 = head

ns = np.arange(5, N_max + 1, 5)
freq_lo, freq_hi = [], []
bayes_lo, bayes_hi = [], []
p_hat_list = []

for n in ns:
    k = flips[:n].sum()
    p_hat = k / n
    p_hat_list.append(p_hat)
    
    # Frequentist 95% CI (normal approximation; Wilson or exact would be better for small n)
    se = np.sqrt(p_hat * (1 - p_hat) / n + 1e-12)
    freq_lo.append(max(0, p_hat - 1.96 * se))
    freq_hi.append(min(1, p_hat + 1.96 * se))
    
    # Bayesian 95% credible interval: Beta(k+1, n-k+1)
    alpha_post = k + 1
    beta_post  = n - k + 1
    lo, hi = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
    bayes_lo.append(lo)
    bayes_hi.append(hi)

freq_lo = np.array(freq_lo)
freq_hi = np.array(freq_hi)
bayes_lo = np.array(bayes_lo)
bayes_hi = np.array(bayes_hi)
p_hat_arr = np.array(p_hat_list)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lo, hi, label, color, title in [
    (axes[0], freq_lo, freq_hi, 'Frequentist 95% CI', C[0], 'Frequentist Confidence Interval'),
    (axes[1], bayes_lo, bayes_hi, 'Bayesian 95% CrI', C[1], 'Bayesian Credible Interval'),
]:
    ax.fill_between(ns, lo, hi, alpha=0.3, color=color, label=label)
    ax.plot(ns, p_hat_arr, lw=1.5, color=color, label='p̂ (MLE)')
    ax.axhline(true_p, color='k', linestyle='--', lw=1.5, label=f'True p={true_p}')
    ax.set_xlabel('N (number of flips)')
    ax.set_ylabel('p')
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Fig 5: Frequentist CI vs Bayesian Credible Interval for Coin Flip (true p=0.6)', y=1.02, fontsize=13)
plt.show()

# Show posterior distributions at a few N values
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
p_vals = np.linspace(0, 1, 300)
for ax, n_idx, n in zip(axes, [2, 9, 39], [15, 50, 200]):
    k = flips[:n].sum()
    prior_pdf = stats.beta.pdf(p_vals, 1, 1)
    post_pdf  = stats.beta.pdf(p_vals, k+1, n-k+1)
    ax.plot(p_vals, prior_pdf, '--', lw=1.5, label='Prior Beta(1,1)', color='grey')
    ax.plot(p_vals, post_pdf,  lw=2,   label=f'Posterior  Beta({k+1},{n-k+1})', color=C[1])
    ax.axvline(true_p, color='k', lw=1.5, linestyle=':', label=f'True p={true_p}')
    ax.set_title(f'N={n}, k={k} heads')
    ax.set_xlabel('p')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.suptitle('Fig 5b: Bayesian Posterior Concentrating as N Grows', y=1.02, fontsize=13)
plt.show()

---
## 6. BIC Model Selection

The **Bayesian Information Criterion** approximates the log marginal likelihood:

$$\text{BIC}(\mathcal{M}_k) = \log p(y \mid X, \hat\theta_{\text{MAP}}, \mathcal{M}_k) - \frac{d_k}{2}\log N$$

where $d_k$ is the number of parameters in model $\mathcal{M}_k$ and $N$ is the dataset size.

**Select the model with the highest BIC** (equivalently, the lowest $-\text{BIC}$, i.e. the penalty-adjusted log-likelihood).

BIC penalizes complexity more aggressively than AIC as $N$ grows (log N vs. constant 2).

**Experiment**: Fit degree 1–8 polynomials to data from a true degree-3 polynomial. BIC should select degree 3 (or nearby).

In [ ]:
# True model: cubic + noise
def true_cubic(x):
    return 2*x**3 - 3*x**2 + x + 0.5

N_bic = 60
noise_bic = 0.4
x_bic = np.linspace(-1, 2, N_bic)
y_bic = true_cubic(x_bic) + np.random.normal(0, noise_bic, N_bic)

max_degree = 8
degrees_bic = range(1, max_degree + 1)
bics, aics, log_liks = [], [], []

for d in degrees_bic:
    # Design matrix for polynomial of degree d
    X_poly = np.vander(x_bic, d + 1, increasing=True)
    
    # Least-squares fit (MLE)
    coeffs, residuals_sq, rank, sv = np.linalg.lstsq(X_poly, y_bic, rcond=None)
    y_hat = X_poly @ coeffs
    
    # MLE estimate of sigma2
    rss = np.sum((y_bic - y_hat)**2)
    sigma2_hat = rss / N_bic
    
    # Log-likelihood at MLE
    log_lik = -N_bic/2 * np.log(2*np.pi*sigma2_hat) - rss / (2*sigma2_hat)
    log_liks.append(log_lik)
    
    # Number of free parameters: d+1 polynomial coeffs + 1 for sigma2
    k_params = d + 1 + 1
    
    bic = log_lik - (k_params / 2) * np.log(N_bic)
    aic = log_lik - k_params
    bics.append(bic)
    aics.append(aic)

bics = np.array(bics)
aics = np.array(aics)

best_bic_deg = list(degrees_bic)[np.argmax(bics)]
best_aic_deg = list(degrees_bic)[np.argmax(aics)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(list(degrees_bic), bics, 'o-', lw=2, label='BIC')
ax.plot(list(degrees_bic), aics, 's--', lw=2, label='AIC')
ax.axvline(best_bic_deg, color=C[0], alpha=0.5, lw=2, linestyle=':', label=f'Best BIC: degree {best_bic_deg}')
ax.axvline(best_aic_deg, color=C[1], alpha=0.5, lw=2, linestyle=':', label=f'Best AIC: degree {best_aic_deg}')
ax.axvline(3, color='k', alpha=0.4, lw=1.5, linestyle='--', label='True degree 3')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('BIC / AIC  (higher = better)')
ax.set_title('Model Selection via BIC and AIC')
ax.legend(fontsize=9)

ax2 = axes[1]
x_fine_bic = np.linspace(-1.2, 2.2, 300)
ax2.scatter(x_bic, y_bic, s=25, alpha=0.7, color='k', label='Data')
ax2.plot(x_fine_bic, true_cubic(x_fine_bic), 'k--', lw=1.5, label='True cubic', alpha=0.5)
for d, ls, col in [(best_bic_deg, '-', C[0]), (1, ':', C[2]), (max_degree, '-.', C[3])]:
    X_plot = np.vander(x_fine_bic, d + 1, increasing=True)
    X_fit  = np.vander(x_bic, d + 1, increasing=True)
    c, *_ = np.linalg.lstsq(X_fit, y_bic, rcond=None)
    ax2.plot(x_fine_bic, X_plot @ c, lw=2, linestyle=ls, color=col, label=f'Degree {d}')
ax2.set_ylim(-5, 10)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Fitted Models (BIC winner, underfitter, overfitter)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Fig 6: BIC Model Selection on Polynomial Regression', y=1.02, fontsize=13)
plt.show()

print(f"True model degree : 3")
print(f"Best BIC degree   : {best_bic_deg}")
print(f"Best AIC degree   : {best_aic_deg}")
print()
print(f"{'Degree':>6}  {'log L':>10}  {'BIC':>10}  {'AIC':>10}")
print('-' * 44)
for d, ll, bic, aic in zip(degrees_bic, log_liks, bics, aics):
    marker = ' ←' if d == best_bic_deg else ''
    print(f"{d:>6}  {ll:>10.2f}  {bic:>10.2f}  {aic:>10.2f}{marker}")

---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| **Loss functions** | MSE is sensitive to outliers (quadratic tail); MAE is robust (linear tail); Huber blends both via threshold δ |
| **Bias-variance** | Low degree → high bias (underfitting); high degree → high variance (overfitting); optimal d minimizes total test error |
| **MLE for Gaussian** | μ̂ = sample mean; σ̂² = biased sample variance (divide by N, not N-1) |
| **MLE = NLL min** | Maximizing log L and minimizing NLL are numerically identical by monotonicity of log |
| **Bayes vs Freq** | Frequentist CI is a procedure guarantee; Bayesian credible interval is a direct probability statement about θ |
| **BIC** | Penalizes complexity by (d/2)log N; selects simpler models than AIC when N is large; approximates log marginal likelihood |

**Next**: Chapter 9 (Linear Regression) applies all these ideas to the supervised regression setting, adding MLE (→ least squares), MAP (→ ridge), and full Bayesian inference (→ predictive distributions).